# Evaluar predicciones: funciones de pérdida en un sistema industrial

En este cuaderno estudiaremos cómo medir si las predicciones de un modelo son buenas o malas. Usaremos un contexto de planta: primero estimaremos consumo eléctrico y después clasificaremos alertas de mantenimiento.

La idea es sencilla: comparar lo que ocurrió realmente con lo que el modelo predijo y convertir esa diferencia en un número que podamos interpretar.

## Objetivos

Al finalizar podrás:

- entender por qué los errores positivos y negativos pueden cancelarse;
- calcular e interpretar MSE y RMSE para valores continuos;
- calcular Binary Cross-Entropy para alertas sí/no;
- calcular entropía cruzada para tres niveles de severidad;
- explicar la diferencia entre pérdida y *accuracy*;
- relacionar problema, activación de salida y función de pérdida;
- observar cómo cambia la pérdida al modificar un parámetro.

## Cómo leer este cuaderno si eres principiante

En todo el cuaderno aparecerán tres cosas:

1. **Dato real:** lo que ocurrió en la planta.
2. **Predicción:** lo que calculó el modelo.
3. **Pérdida:** un número que resume qué tan lejos estuvo la predicción.

La regla general es: **una pérdida más pequeña significa un error menor**, siempre que comparemos el mismo tipo de problema. No compares directamente un MSE con una BCE: usan escalas y preguntas diferentes.

No es necesario memorizar las fórmulas todavía. Primero entiende qué representa cada valor y después observa cómo la fórmula convierte esos valores en un resultado.

## Glosario básico

- **Modelo:** una regla que produce una predicción a partir de datos.
- **Dato real:** la respuesta que observamos o medimos.
- **Predicción:** la respuesta calculada por el modelo.
- **Error:** la distancia entre el dato real y la predicción.
- **Pérdida:** una forma de convertir los errores en un número para comparar modelos.
- **Probabilidad:** un número entre 0 y 1. Por ejemplo, 0.80 equivale a 80% de confianza.
- **Clase:** una categoría, como “alerta” o “normal”.
- **Accuracy:** proporción de decisiones correctas.

En los ejemplos, 1 normalmente significa “sí ocurrió” y 0 significa “no ocurrió”.

## Importación de herramientas

Usaremos NumPy para operar con números, Pandas para construir tablas y Matplotlib para dibujar. Estas librerías no realizan el aprendizaje por sí mismas: aquí las usamos para hacer visibles las ideas matemáticas.

### Contexto para principiantes

Antes de calcular una pérdida necesitamos una forma ordenada de guardar los datos. En vez de trabajar número por número, usaremos arreglos: son como pequeñas listas que permiten hacer la misma operación sobre varios valores al mismo tiempo.


### Antes de ejecutar este bloque

Primero se cargan las herramientas. Todavía no estamos entrenando ningún modelo.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Importamos las librerías que necesitamos.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

### ¿Cómo leer esta parte?

No esperamos una respuesta numérica todavía. Si la celda termina sin mostrar un error, significa que las herramientas quedaron disponibles y podemos continuar.


### ¿Qué hace este código?

Se importan tres librerías. NumPy trabaja con arreglos numéricos, Pandas organiza resultados en tablas y Matplotlib crea gráficas. Las dos últimas líneas solo configuran la presentación; no cambian los cálculos.

## 1. El error puede ocultarse al promediarlo

Supongamos que un motor consumió 48 kWh en dos intervalos. En uno el modelo se quedó corto y en el otro se pasó por la misma cantidad.

Definimos el residuo como:

$$e = y - \hat{y}$$

Si promediamos residuos con signo, un error positivo puede cancelar a uno negativo.

### Antes de ejecutar

Aquí guardamos las mediciones reales y las estimaciones. La resta muestra el error de cada intervalo y la tabla permite revisarlo antes de resumirlo.


### Contexto para principiantes

Aquí simulamos dos momentos de operación de un motor. El valor real es lo que midió el medidor; la estimación es lo que calculó el modelo. Comparar ambos valores nos dice qué tan lejos estuvo el pronóstico.


### Antes de ejecutar este bloque

Entrada: dos consumos reales y dos estimaciones. Salida: una tabla con los errores.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Guardamos primero lo que ocurrió y después lo que el modelo estimó.
consumo_real = np.array([48, 48])
consumo_estimado = np.array([44, 52])
residuos = consumo_real - consumo_estimado

tabla_residuos = pd.DataFrame({
    "Consumo real (kWh)": consumo_real,
    "Estimación (kWh)": consumo_estimado,
    "Residuo": residuos,
    "Residuo²": residuos ** 2,
})
display(tabla_residuos)
print(f"Promedio de residuos: {residuos.mean():.2f}")
print(f"Promedio de residuos²: {np.mean(residuos ** 2):.2f}")

### ¿Cómo leer la tabla?

Mira primero la columna Residuo. Un número positivo indica que el valor real fue mayor que la estimación; uno negativo indica que la estimación quedó por encima. El promedio de residuos puede ser engañoso porque los signos se cancelan. Residuo² siempre es positivo y por eso permite comparar el tamaño del error.


### Interpretación sencilla

El promedio de residuos es 0, pero ninguna estimación es exacta. El +4 y el -4 se cancelan. Al elevarlos al cuadrado, ambos se convierten en 16 y los dos errores quedan visibles. Esta es la razón principal para usar una pérdida cuadrática.

## 2. MSE y RMSE para consumo eléctrico

El **Mean Squared Error** o MSE es el promedio de los errores al cuadrado:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2$$

El RMSE es la raíz cuadrada del MSE:

$$RMSE = \sqrt{MSE}$$

MSE está en unidades cuadradas; RMSE vuelve a las unidades originales, en este caso kWh.

### Antes de ejecutar

Primero construimos una función para no repetir la fórmula del MSE. Después comparamos dos pronósticos y calculamos también RMSE, que se expresa nuevamente en kWh.


### Contexto para principiantes

Una función es una receta con nombre. La función mse recibe dos listas: una con la respuesta correcta y otra con la predicción. La línea return entrega un solo número: el promedio de los errores al cuadrado. Luego usamos la misma receta para dos pronósticos distintos.


### Antes de ejecutar este bloque

Entrada: valores reales y predicciones. Salida: un número que resume el error cuadrático.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Esta función calcula el promedio de los errores al cuadrado.
def mse(y, prediccion):
    return np.mean((y - prediccion) ** 2)

consumo_real = np.array([32, 46, 61, 75])
pronostico_a = np.array([29, 50, 57, 71])
pronostico_b = np.array([33, 44, 63, 74])

tabla_mse = pd.DataFrame({
    "Real": consumo_real,
    "Pronóstico A": pronostico_a,
    "Error² A": (consumo_real - pronostico_a) ** 2,
    "Pronóstico B": pronostico_b,
    "Error² B": (consumo_real - pronostico_b) ** 2,
})
display(tabla_mse)
mse_a, mse_b = mse(consumo_real, pronostico_a), mse(consumo_real, pronostico_b)
print(f"MSE A: {mse_a:.2f}")
print(f"MSE B: {mse_b:.2f}")
print(f"RMSE A: {np.sqrt(mse_a):.2f} kWh")
print(f"RMSE B: {np.sqrt(mse_b):.2f} kWh")

### ¿Cómo decidir cuál es mejor?

Compara MSE A con MSE B: el menor gana dentro de este ejemplo. Después observa RMSE, porque está en kWh y es más fácil de comunicar. MSE sirve para hacer la comparación matemática; RMSE ayuda a explicar el resultado a una persona que conoce las unidades del problema.


### Interpretación sencilla

El pronóstico con MSE menor comete errores cuadrados más pequeños en promedio. RMSE permite decir algo como “el error típico ronda los 2 kWh”. No significa que todas las predicciones fallen exactamente por esa cantidad; es un resumen del conjunto completo.

### ¿Por qué el cuadrado castiga los errores grandes?

Un error de 6 kWh pesa mucho más que uno de 2 kWh porque se eleva al cuadrado. Esto es útil cuando una desviación grande debe preocuparnos especialmente.

### Antes de ejecutar

Generamos muchos tamaños posibles de error y calculamos su cuadrado. La gráfica nos permite ver de manera visual por qué los errores grandes pesan más.


### Contexto para principiantes

No estamos usando datos de un motor en esta gráfica. Solo preguntamos: si el error mide 0, 1, 2, 3 o más kWh, ¿qué valor produce al elevarlo al cuadrado? Esto permite entender el comportamiento de MSE de forma aislada.


### Antes de ejecutar este bloque

Entrada: tamaños posibles de error. Salida: una gráfica que muestra cómo crece el error al cuadrado.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Creamos muchos valores para poder dibujar una curva suave.
magnitudes = np.linspace(0, 7, 140)
perdida_cuadratica = magnitudes ** 2
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(magnitudes, perdida_cuadratica, color="#167a72", linewidth=2.5)
ax.scatter([1, 3, 6], [1, 9, 36], color="#d95f02", s=65)
ax.set_title("La penalización cuadrática crece con rapidez")
ax.set_xlabel("Magnitud del error (kWh)")
ax.set_ylabel("Error²")
plt.show()

### ¿Qué debemos observar?

La curva se hace cada vez más inclinada. Cerca de cero, aumentar un poco el error cambia poco la pérdida; con errores grandes, el mismo aumento produce un cambio mucho mayor. Por eso MSE es sensible a situaciones donde una predicción se aleja demasiado.


### Interpretación sencilla

Un error de 6 produce 36, mientras que un error de 1 produce 1. Por eso MSE da mucho peso a los errores grandes. La desventaja es que una lectura atípica puede influir bastante en el promedio.

## 3. Binary Cross-Entropy para alertas

Ahora el modelo estima la probabilidad de que una inspección requiera atención inmediata. Para una etiqueta real $y$ y una probabilidad $p$ de alerta usamos:

$$L = -\left[y\log(p)+(1-y)\log(1-p)\right]$$

Si la alerta ocurrió, interesa que $p$ sea cercana a 1. Si no ocurrió, interesa que $p$ sea cercana a 0.

### Antes de ejecutar

La función recibe la etiqueta real y la probabilidad de alerta. Recuerda: 1 significa alerta y 0 significa normal. El recorte evita problemas al calcular logaritmos de cero.


### Contexto para principiantes

En un problema binario solo existen dos respuestas: ocurrió una alerta o no ocurrió. La variable y guarda la respuesta real como 1 o 0. La variable p guarda la confianza del modelo, por ejemplo 0.90 significa “creo que hay un 90% de probabilidad de alerta”.


### Antes de ejecutar este bloque

Entrada: etiqueta real y probabilidad. Salida: el costo de una sola predicción binaria.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Calculamos el costo de una predicción de alerta.
def bce_individual(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

casos = pd.DataFrame({
    "Alerta real": [1, 1, 0, 0],
    "Probabilidad de alerta": [0.92, 0.58, 0.18, 0.72],
})
casos["BCE"] = [bce_individual(y, p) for y, p in zip(casos["Alerta real"], casos["Probabilidad de alerta"])]
display(casos)
print(f"BCE promedio: {casos['BCE'].mean():.4f}")

### ¿Cómo leer cada BCE?

Cuando la etiqueta real es 1, una probabilidad alta es buena. Cuando la etiqueta real es 0, una probabilidad baja es buena. La BCE no se interpreta como un porcentaje; es un costo matemático y, para el mismo problema, un valor menor indica una predicción probabilística mejor.


### Interpretación sencilla

La BCE penaliza mucho una predicción equivocada y muy segura. Decir “hay alerta” con probabilidad 0.99 cuando el caso era normal es peor que decirlo con 0.55. Una probabilidad cercana a 0.50 es prudente, pero también poco informativa.

### Forma de la Binary Cross-Entropy

La siguiente gráfica muestra la pérdida para todas las probabilidades posibles. Así podemos ver qué ocurre cuando el modelo se acerca a la respuesta correcta o se aleja de ella.

### Antes de ejecutar

Probamos muchas probabilidades para dibujar las dos curvas de BCE. Así veremos qué ocurre cuando el modelo se acerca a la respuesta correcta o se aleja de ella.


### Contexto para principiantes

Una sola tabla muestra algunos ejemplos, pero una gráfica nos deja observar todo el recorrido de la probabilidad. Probamos valores desde casi 0 hasta casi 1 y calculamos la pérdida que correspondería a cada uno.


### Antes de ejecutar este bloque

Entrada: muchas probabilidades. Salida: dos curvas, una para alerta y otra para caso normal.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Revisamos muchas probabilidades, desde casi 0 hasta casi 1.
p_eje = np.linspace(0.001, 0.999, 500)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(p_eje, bce_individual(1, p_eje), label="Etiqueta real: alerta", linewidth=2.3)
ax.plot(p_eje, bce_individual(0, p_eje), label="Etiqueta real: normal", linewidth=2.3)
ax.set_title("Binary Cross-Entropy según la probabilidad")
ax.set_xlabel("Probabilidad de alerta")
ax.set_ylabel("Pérdida")
ax.legend()
plt.show()

### ¿Cómo leer las curvas?

La curva de alerta real baja hacia la derecha: cuanto más cerca esté la probabilidad de 1, mejor. La curva de caso normal baja hacia la izquierda: cuanto más cerca esté de 0, mejor. En los extremos equivocados la pérdida crece mucho porque el modelo estaba muy seguro y se equivocó.


### Interpretación sencilla

La pérdida es pequeña cuando la probabilidad favorece a la clase correcta. Crece rápidamente cuando el modelo está muy seguro de la clase equivocada. Por eso BCE sirve cuando la confianza de la predicción también importa.

## 4. Pérdida frente a *accuracy*

La *accuracy* solo pregunta cuántas decisiones fueron correctas. Para convertir una probabilidad en clase usaremos el umbral 0.5. Dos modelos pueden acertar los mismos casos y, aun así, tener probabilidades de distinta calidad.

### Antes de ejecutar

Estas funciones resumen la BCE y convierten probabilidades en clases usando 0.5 como umbral. Luego podemos comparar aciertos y calidad de confianza.


### Contexto para principiantes

Aquí comparamos dos sistemas. Primero transformamos cada probabilidad en una decisión usando 0.5: por encima decimos “alerta” y por debajo decimos “normal”. Después calculamos BCE para no perder la información de confianza que había antes de tomar esa decisión.


### Antes de ejecutar este bloque

Entrada: etiquetas y probabilidades. Salida: accuracy y BCE para comparar dos sistemas.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Accuracy cuenta aciertos; BCE revisa también la confianza.
def bce_promedio(y, p):
    return np.mean(bce_individual(y, p))

def accuracy_binaria(y, p, umbral=0.5):
    return np.mean((p >= umbral).astype(int) == y)

y_alerta = np.array([1, 0, 1, 0, 1])
prob_moderadas = np.array([0.55, 0.45, 0.62, 0.40, 0.58])
prob_separadas = np.array([0.88, 0.08, 0.76, 0.15, 0.93])

comparacion = pd.DataFrame({
    "Sistema": ["Probabilidades moderadas", "Probabilidades separadas"],
    "Accuracy": [accuracy_binaria(y_alerta, prob_moderadas), accuracy_binaria(y_alerta, prob_separadas)],
    "BCE": [bce_promedio(y_alerta, prob_moderadas), bce_promedio(y_alerta, prob_separadas)],
})
display(comparacion)

### ¿Por qué mirar dos columnas?

Accuracy es fácil de entender: indica la fracción de respuestas correctas. BCE agrega otra pregunta: ¿las probabilidades estaban bien orientadas y eran razonables? Un sistema puede acertar, pero hacerlo con probabilidades muy cercanas a 0.5; otro puede acertar con mayor seguridad.


### Interpretación sencilla

La *accuracy* convierte probabilidades en respuestas de sí o no y pierde detalle. BCE distingue entre acertar apenas por encima de 0.5 y acertar con una probabilidad mucho mayor. Esto puede ayudar a ordenar inspecciones y priorizar recursos.

## 5. Entropía cruzada para tres niveles

Una alerta puede ser **preventiva**, **programada** o **urgente**. Con *softmax*, las probabilidades de las tres clases suman 1. Para un objetivo *one-hot*:

$$L=-\sum_{k=1}^{K}y_k\log(\hat{y}_k)$$

En la práctica, esta suma termina seleccionando la probabilidad de la clase verdadera.

### Antes de ejecutar

Cada arreglo contiene las probabilidades de los tres niveles de alerta. argmax identifica la clase con mayor probabilidad y la fórmula evalúa la probabilidad de la clase verdadera.


### Contexto para principiantes

Ahora hay tres respuestas posibles en vez de dos. Cada fila contiene tres probabilidades que suman 1, como una distribución de confianza. Por ejemplo, [0.10, 0.20, 0.70] significa que el modelo considera más probable la tercera clase.


### Antes de ejecutar este bloque

Entrada: tres probabilidades por modelo. Salida: clase elegida y pérdida para cada modelo.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Cada arreglo contiene una probabilidad para cada nivel.
niveles = ["preventiva", "programada", "urgente"]
indice_real = 2
predicciones = {
    "A": np.array([0.08, 0.17, 0.75]),
    "B": np.array([0.20, 0.35, 0.45]),
    "C": np.array([0.10, 0.80, 0.10]),
}
filas = []
for nombre, p in predicciones.items():
    filas.append({
        "Modelo": nombre,
        "Clase elegida": niveles[np.argmax(p)],
        "Pérdida": -np.log(p[indice_real]),
    })
display(pd.DataFrame(filas))

### ¿Cómo leer el resultado?

Clase elegida es la opción con el número más grande. Pérdida, en cambio, revisa cuánto valor recibió la respuesta que realmente ocurrió. Por eso un modelo puede elegir correctamente una clase y aun así tener una pérdida relativamente alta si su confianza fue baja.


### Interpretación sencilla

La pérdida revisa la probabilidad que cada modelo dio a la clase verdadera, que aquí es “urgente”. A tiene 0.75 y por eso su pérdida es menor. B todavía elige urgente, pero con menos confianza. C se inclina por una clase incorrecta y recibe una penalización alta.

### Etiquetas *one-hot* y etiquetas enteras

Podemos escribir “urgente” como [0, 0, 1] o como el índice 2. Ambas formas representan la misma respuesta.

### Antes de ejecutar

Calculamos la misma pérdida usando dos formas de guardar la etiqueta: un vector one-hot y un índice entero. Los resultados deben coincidir.


### Contexto para principiantes

Una etiqueta puede guardarse de dos maneras. One-hot usa un 1 en la posición correcta y ceros en las demás. La etiqueta entera usa solo la posición: aquí 2 representa la tercera clase. Son dos formatos para expresar la misma respuesta.


### Antes de ejecutar este bloque

Entrada: una misma respuesta en dos formatos. Salida: dos pérdidas que deben coincidir.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Comparamos dos formas de guardar la misma etiqueta.
objetivo_one_hot = np.array([0, 0, 1])
objetivo_entero = 2
probabilidades = np.array([0.12, 0.23, 0.65])
perdida_one_hot = -np.sum(objetivo_one_hot * np.log(probabilidades))
perdida_entera = -np.log(probabilidades[objetivo_entero])
print(f"Pérdida con one-hot: {perdida_one_hot:.6f}")
print(f"Pérdida con índice entero: {perdida_entera:.6f}")

### ¿Qué significa que coincidan?

Que ambas representaciones producen el mismo número demuestra que no cambiamos la idea de la pérdida; solo cambiamos la forma de guardar la etiqueta. Esto será importante al usar bibliotecas de aprendizaje automático, porque debemos elegir la función compatible con el formato de nuestros datos.


### Interpretación sencilla

Los dos cálculos deben dar el mismo resultado. La diferencia está en cómo guardamos la etiqueta, no en lo que queremos medir. En ambos casos importa la probabilidad asignada a la clase correcta.

## 6. Compatibilidad entre problema y pérdida

| Problema | Activación final | Objetivo | Pérdida habitual |
|---|---|---|---|
| Consumo continuo | Lineal | Número | MSE o MAE |
| Alerta sí/no | Sigmoid | 0 o 1 | Binary Cross-Entropy |
| Severidad exclusiva | Softmax | Vector o índice | Categorical Cross-Entropy |

La pérdida debe corresponder a la forma de la salida y al tipo de respuesta que queremos predecir.

## 7. La pérdida como función de un parámetro

Usaremos un modelo muy pequeño:

$$\hat{y}=wx+b$$

Fijaremos $b=0$ y probaremos diferentes valores de $w$. La curva nos mostrará qué pendiente produce el menor MSE para estas mediciones.

### Antes de ejecutar

Probamos muchas pendientes para un modelo muy pequeño. En cada prueba calculamos el MSE y buscamos el punto donde la pérdida es menor.


### Contexto para principiantes

Este es un modelo deliberadamente pequeño: multiplica la carga por una pendiente w. Probamos muchas pendientes, como si giráramos una perilla, y medimos cuánto se alejan las predicciones de los consumos medidos.


### Antes de ejecutar este bloque

Entrada: carga y consumo medido. Salida: una curva que indica qué pendiente produce menor error.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Probamos diferentes valores para la pendiente del modelo.
carga = np.array([1, 2, 3, 4, 5])
consumo_medido = np.array([3, 6, 9, 12, 15])
pendientes = np.linspace(0, 5, 251)
perdidas = np.array([mse(consumo_medido, w * carga) for w in pendientes])
indice = np.argmin(perdidas)
w_optimo = pendientes[indice]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(pendientes, perdidas, color="#5e3c99", linewidth=2.5)
ax.scatter(w_optimo, perdidas[indice], color="#e66101", s=70, label=f"Menor MSE: w = {w_optimo:.2f}")
ax.set_title("Pérdida según la pendiente del modelo")
ax.set_xlabel("Pendiente w")
ax.set_ylabel("MSE")
ax.legend()
plt.show()
print(f"Pendiente seleccionada: {w_optimo:.2f}")
print(f"MSE mínimo: {perdidas[indice]:.4f}")

### ¿Cómo leer la gráfica?

El eje horizontal contiene posibles pendientes y el vertical contiene el MSE. El punto más bajo es la mejor elección entre las que probamos. La búsqueda no “adivina” mágicamente: compara opciones una por una y conserva la que tiene menor costo.


### Interpretación sencilla

Cada punto de la curva representa una prueba distinta para la pendiente. El punto más bajo es la opción que produce menor error en estos datos. En un modelo real habría muchos parámetros y usaríamos métodos como descenso por gradiente para buscar eficientemente.

## 8. Actividad aplicada

Modifica las probabilidades de los dos sistemas. Intenta conservar la misma *accuracy* y reducir la BCE. Después explica cuál sistema sería más útil si una alerta activa una inspección costosa.

### Antes de ejecutar

Esta es una actividad editable. Cambia las probabilidades, ejecuta de nuevo y observa que accuracy y BCE responden a aspectos diferentes.


### Contexto para principiantes

Esta celda es un pequeño experimento. No necesitas cambiar la fórmula; solo puedes cambiar las probabilidades y observar cómo responde el resultado. Es una forma segura de construir intuición antes de trabajar con modelos más complejos.


### Antes de ejecutar este bloque

Entrada: etiquetas y probabilidades que puedes modificar. Salida: una tabla para practicar.

Lee el código de arriba hacia abajo. Cada línea prepara un dato, hace una operación o muestra un resultado.


In [ ]:
# Cambia estas probabilidades para experimentar.
y_actividad = np.array([1, 0, 0, 1, 1])

# Modifica estas dos series y ejecuta de nuevo la celda.
probabilidades_1 = np.array([0.60, 0.40, 0.45, 0.58, 0.65])
probabilidades_2 = np.array([0.91, 0.12, 0.06, 0.84, 0.95])

actividad = pd.DataFrame({
    "Sistema": ["Estimaciones moderadas", "Estimaciones decididas"],
    "Accuracy": [accuracy_binaria(y_actividad, probabilidades_1), accuracy_binaria(y_actividad, probabilidades_2)],
    "BCE": [bce_promedio(y_actividad, probabilidades_1), bce_promedio(y_actividad, probabilidades_2)],
})
display(actividad)

### Pasos sugeridos

Primero ejecuta la celda sin cambiar nada. Después modifica una probabilidad correcta de 0.60 a 0.90 y observa la BCE. Luego cambia una probabilidad hasta cruzar 0.50 y observa la accuracy. Así notarás que BCE cambia poco a poco, mientras accuracy puede cambiar de golpe.


### Interpretación sencilla

Con los valores iniciales, ambos sistemas aciertan. El segundo separa mejor alertas y casos normales, por eso su BCE es menor. Observa que la *accuracy* puede permanecer igual mientras la BCE cambia gradualmente. En un proyecto real también revisaríamos falsos negativos, calibración y datos nuevos.

## Cierre

MSE resume errores de valores continuos y castiga especialmente los errores grandes. BCE evalúa probabilidades binarias y penaliza la confianza mal colocada. La entropía cruzada categórica hace lo mismo cuando existen varias clases.

La *accuracy* responde “¿acertó o no?”, mientras que la pérdida también puede responder “¿qué tan buena fue la confianza?”. Finalmente, vimos que modificar un parámetro cambia las predicciones y, con ellas, la pérdida.

## Preguntas para discutir

1. ¿Por qué un promedio de errores con signo puede dar una impresión equivocada?
2. ¿En qué situación preferirías RMSE en lugar de MSE para comunicar resultados?
3. ¿Por qué dos modelos con la misma *accuracy* pueden tener distinta BCE?
4. ¿Qué significa que una predicción tenga alta confianza pero una pérdida grande?
5. ¿Por qué no tiene sentido comparar directamente un MSE con una BCE solo por su tamaño numérico?